In [1]:
!pip install -q cassio datasets langchain openai tiktoken faiss-cpu
!pip install -q langchain-community
!pip install -q langchain-openai
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from PyPDF2 import PdfReader

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [3]:
# Read PDF file
from google.colab import drive
drive.mount('/gdrive/')
pdfreader = PdfReader('/gdrive/MyDrive/OrchidPharmed/My paper.pdf')
raw_text = ''
for i, page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        raw_text += content

with open("sample.txt", "w", encoding="utf-8") as f:
    f.write(raw_text)

Mounted at /gdrive/


In [4]:
# Create chunks from the text file
loader = TextLoader("sample.txt", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 100,
)

chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")


Split into 59 chunks.


In [5]:
# Create an embedding and LLM instance from openAI
embeddings = OpenAIEmbeddings(api_key="sk-None-UYzbbBzqVwU97qyzRG7LT3BlbkFJzt2QR0sm9Nyhd89S7YUA", model="text-embedding-3-small")
sample_embdedings = embeddings.embed_query("Hello, world!")
print(f"length of sample embedding: {len(sample_embdedings)}, \nfirst 5 values: \n{sample_embdedings[:5]}")

llm = ChatOpenAI(model="gpt-4o", temperature=0, openai_api_key="sk-None-UYzbbBzqVwU97qyzRG7LT3BlbkFJzt2QR0sm9Nyhd89S7YUA")
smaple_response = llm.invoke("Hello, world!")
print(f"\n\nSample response from LLM: \n{smaple_response}")

length of sample embedding: 1536, 
first 5 values: 
[-0.019143931567668915, -0.025292053818702698, -0.0017211713129654527, 0.01883450709283352, -0.03382139280438423]


Sample response from LLM: 
content='Hello! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 11, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-Cv0gHxtBscu3sGf74Tr8sXMwe6HWp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019b9354-8487-7f90-a6cf-03120d800e5a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 11, 'output_tokens': 9, 'total_tokens': 20, 'input_token_details': {'audio': 0

In [6]:
# Create FAISS vectorestore, and save it locally. Once saved, you can bypass the creation step next time.
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Created FAISS vectorstore with {len(chunks)} vectors.")

vectorstore.save_local("faiss_index")

Created FAISS vectorstore with 59 vectors.


In [7]:
# Load FAISS vectorestore from local disk
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [9]:
# test similarity search

query = "One-minute segments in these recordings are determined by a SA specialist as normal or an apnea event. "

results = vectorstore.similarity_search(
    query,
    k=10,
)

print(results)
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)
    print("Source:", doc.metadata.get("source"))



[Document(id='9d7d3082-730b-4f98-9f9a-72eecf4cd8b4', metadata={'source': 'sample.txt'}, page_content='saturation).\nOne-minute segments in these recordings are determined \nby a SA specialist as normal or an apnea event. Generally, \nthere are 6514 segments of apnea and 10,496 segments of \nnormal class. Each of these recordings is an ECG signal with \na sampling frequency of 100\xa0Hz and a resolution of 16 bits \nfor a period of about 8 to 9\xa0h. In addition, each recording \nof ECG signals is different according to some parameters \nsuch as age, sex, weight, height, apnea index (AI), hypopnea \nindex (HI), apnea–hypopnea index (AHI).\n2.2  Pre‑processing\nIn this paper, we use the standard steps in preprocessing. \nFirst, the 1-min ECG signals are divided into two-second \nsegments. Then the ECG signal is filtered to remove the \nnoise using a 3rd order Butterworth filter. One example\xa0of Downloaded from https://iranpaper.ir\nhttps://www.tarjomano.com\nhttps://www.tarjomano.com\n

In [10]:
# conver vectorstore to retriver and test it
retriver = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":5})
retriver.invoke(query)

[Document(id='9d7d3082-730b-4f98-9f9a-72eecf4cd8b4', metadata={'source': 'sample.txt'}, page_content='saturation).\nOne-minute segments in these recordings are determined \nby a SA specialist as normal or an apnea event. Generally, \nthere are 6514 segments of apnea and 10,496 segments of \nnormal class. Each of these recordings is an ECG signal with \na sampling frequency of 100\xa0Hz and a resolution of 16 bits \nfor a period of about 8 to 9\xa0h. In addition, each recording \nof ECG signals is different according to some parameters \nsuch as age, sex, weight, height, apnea index (AI), hypopnea \nindex (HI), apnea–hypopnea index (AHI).\n2.2  Pre‑processing\nIn this paper, we use the standard steps in preprocessing. \nFirst, the 1-min ECG signals are divided into two-second \nsegments. Then the ECG signal is filtered to remove the \nnoise using a 3rd order Butterworth filter. One example\xa0of Downloaded from https://iranpaper.ir\nhttps://www.tarjomano.com\nhttps://www.tarjomano.com\n

In [11]:
# Now we create a retrieval-augmented generation (RAG) chain using the LLM and the vectorstore as the retriever.

prompt = PromptTemplate.from_template(
    "You are a question answering assistant. You are given a context and a question. You should only answer the question based on the context.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "If the question cannot be answered using the context, respond with 'I don't know.'\n\n"
    "Answer:"
)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":5})

qa_chain = (
    {
        "context": (lambda x: x["question"]) | retriever,   # retriever is now passed the question string directly
        "question": RunnablePassthrough() # Pass the original question through
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [13]:
# Finally, we can ask a question and get an answer.
query = "What are cnn models used in this paper?"
answer = qa_chain.invoke({"question": query})
print(f"\n\nFinal answer:\n{answer}")



Final answer:
The CNN models used in this paper are Efficient Net-B0, Efficient Net-B1, Efficient Net-B2, Inception-v3, and Xception.
